**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 15 · 大模型基础：采样、对齐训练与奖励过优化

**心理大模型评测与 Agent** · Agent 与模型 · 建议用时 3 到 4 小时

**今天的目标**

- 解释逐词生成和 temperature、top-p 采样，知道评测时为什么要固定这些参数
- 说清预训练、SFT、RLHF、DPO 各做什么，写出 DPO 的损失函数
- 把奖励模型理解为一个“裁判”，解释奖励过优化
- 说明附和倾向为什么在心理支持场景中特别危险

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. temperature 与 top-p

六个可能的回复开头词及其分数（logit）。

In [ ]:
tokens = ["It sounds", "I hear", "You should", "Have you", "Try to", "Everyone"]
logits = np.array([2.2, 1.8, 1.5, 1.2, 0.6, -0.5])
def softmax_T(z, T):
    e = np.exp((z - z.max()) / T)
    return e / e.sum()
pd.DataFrame({T: softmax_T(logits, T) for T in [0.3, 1.0, 1.8]}, index=tokens).round(3)

**练习 15.1**:T = 0.5 时，“It sounds”的概率是多少？

In [ ]:
ans_15_1 = softmax_T(logits, 0.5)[0]
check("练习 15.1", ans_15_1, softmax_T(logits, 0.5)[0], tol=1e-6)

**练习 15.2**：写 `top_p_set(probs, p)`，返回按概率从高到低、累计概率首次达到 p 时包含的词数。T = 1、p = 0.9 时保留几个词？

In [ ]:
def top_p_set(probs, p):
    s = np.sort(probs)[::-1]
    return int(np.searchsorted(np.cumsum(s), p) + 1)

ans_15_2 = top_p_set(softmax_T(logits, 1.0), 0.9)
_order = np.sort(softmax_T(logits, 1.0))[::-1]
check("练习 15.2", ans_15_2, int(np.searchsorted(np.cumsum(_order), 0.9) + 1))

## 2. 奖励过优化：best-of-n

每条候选回复有一个基础质量 `base` 和一个长度 `L`（标准化）。真实质量对过长或过短的回复打折扣；奖励模型却偏好更长的回复。我们从 n 条候选里选奖励最高的一条。

In [ ]:
rng = np.random.default_rng(0)
ns = [1, 2, 4, 8, 16, 32, 64, 128, 256]
rows = []
for n in ns:
    base = rng.normal(0, 1, (3000, n)); L = rng.normal(0, 1, (3000, n))
    true = base + 0.3 * L - 0.25 * L ** 2
    proxy = base + 0.8 * L
    pick = proxy.argmax(1)
    rows.append({"n": n, "proxy": proxy[np.arange(3000), pick].mean(), "true": true[np.arange(3000), pick].mean(),
                 "length": L[np.arange(3000), pick].mean()})
bon = pd.DataFrame(rows); bon

**练习 15.3**：真实质量在哪个 n 达到最高？

In [ ]:
ans_15_3 = int(bon.loc[bon.true.idxmax(), "n"])
check("练习 15.3", ans_15_3, int(bon.loc[bon.true.idxmax(), "n"]))

## 3. DPO 损失的一次计算

一个偏好对：被选中回答 y_w 是温和挑战，被拒绝回答 y_l 是附和。对数概率如下，β = 0.1。

In [ ]:
logp_theta_w, logp_ref_w = -42.0, -45.0
logp_theta_l, logp_ref_l = -40.0, -39.0
beta = 0.1
sigmoid = lambda x: 1 / (1 + np.exp(-x))

**练习 15.4**：计算 DPO 损失。

In [ ]:
ans_15_4 = -np.log(sigmoid(beta * ((logp_theta_w - logp_ref_w) - (logp_theta_l - logp_ref_l))))
_m = beta * ((logp_theta_w - logp_ref_w) - (logp_theta_l - logp_ref_l))
check("练习 15.4 DPO 损失", ans_15_4, -np.log(sigmoid(_m)), tol=1e-6)

*写一句话：这个损失值说明当前模型相对参考模型，是更偏向温和挑战还是更偏向附和？*

## 口述题（录音，每题 90 秒）

1. RLHF 和 DPO 的区别是什么？奖励过优化为什么和评测有关？
   <details><summary>参考要点</summary>RLHF 先训练奖励模型再做强化学习，DPO 直接用偏好对优化策略。奖励模型本身是一个裁判，有偏差时，优化会放大偏差，代理分数升高而真实质量下降。所以训练信号的测量质量要先检验。</details>

## 研究延伸

心理支持场景下的偏好数据应该由谁来标注？咨询师和来访者的偏好不一致时怎么办？可以设计一项研究：比较咨询师偏好与用户偏好训练出的奖励模型，在附和倾向上的差异。

In [ ]:
summary()